In [1]:
import numpy as np
from collections import defaultdict, Counter
import pandas as pd

In [2]:
sentences = [
    "Mary Jane can see Will.",
    "Spot will see Mary.",
    "Will Jane spot Mary?",
    "Mary will pat Spot."
]

df = pd.DataFrame({"Sentence": sentences})
print(df)

                  Sentence
0  Mary Jane can see Will.
1      Spot will see Mary.
2     Will Jane spot Mary?
3      Mary will pat Spot.


In [3]:
train_data = [
    [("Mary", "NNP"), ("Jane", "NNP"), ("can", "MD"), ("see", "VB"), ("Will", "NNP"), (".", ".")],
    [("Spot", "NNP"), ("will", "MD"), ("see", "VB"), ("Mary", "NNP"), (".", ".")],
    [("Will", "MD"), ("Jane", "NNP"), ("spot", "VB"), ("Mary", "NNP"), ("?", ".")],
    [("Mary", "NNP"), ("will", "MD"), ("pat", "VB"), ("Spot", "NNP"), (".", ".")]
]

In [4]:
transition_counts = defaultdict(Counter)
emission_counts = defaultdict(Counter)
tag_counts = Counter()

for sentence in train_data:
    prev_tag = "<s>"
    for word, tag in sentence:
        transition_counts[prev_tag][tag] += 1
        emission_counts[tag][word.lower()] += 1
        tag_counts[tag] += 1
        prev_tag = tag


In [5]:
transition_probs = defaultdict(dict)
emission_probs = defaultdict(dict)

for prev_tag in transition_counts:
    total = sum(transition_counts[prev_tag].values())
    for tag in transition_counts[prev_tag]:
        transition_probs[prev_tag][tag] = transition_counts[prev_tag][tag] / total

for tag in emission_counts:
    total = sum(emission_counts[tag].values())
    for word in emission_counts[tag]:
        emission_probs[tag][word] = emission_counts[tag][word] / total

tags = list(tag_counts.keys())


In [12]:

def viterbi(sentence):
    words = sentence.lower().split()
    V = [{}]
    path = {}

    for tag in tags:
        trans = transition_probs["<s>"].get(tag, 1e-6)
        emit = emission_probs[tag].get(words[0], 1e-6)
        V[0][tag] = trans * emit
        path[tag] = [tag]

    for t in range(1, len(words)):
        V.append({})
        new_path = {}

        for curr_tag in tags:
            (prob, prev_state) = max(
                (V[t-1][prev_tag] *
                 transition_probs[prev_tag].get(curr_tag, 1e-6) *
                 emission_probs[curr_tag].get(words[t], 1e-6),
                 prev_tag)
                for prev_tag in tags
            )

            V[t][curr_tag] = prob
            new_path[curr_tag] = path[prev_state] + [curr_tag]

        path = new_path

    n = len(words) - 1
    (prob, state) = max((V[n][tag], tag) for tag in tags)

    return path[state]


In [17]:
test_data = [
    [("Will", "NNP"), ("can", "MD"), ("spot", "VB"), ("Mary", "NNP")],
    [("Mary", "NNP"), ("will", "MD"), ("see", "VB"), ("Spot", "NNP")]
]

total_correct = 0
total_tokens = 0

all_actual = []
all_predicted = []

for sentence in test_data:
    words = [word for word, tag in sentence]
    actual_tags = [tag for word, tag in sentence]

    sentence_str = " ".join(words)
    predicted_tags = viterbi(sentence_str)

    print("\nSentence:", sentence_str)
    print("Actual:", actual_tags)
    print("Predicted:", predicted_tags)

    all_actual.extend(actual_tags)
    all_predicted.extend(predicted_tags)

    total_correct += sum(p == g for p, g in zip(predicted_tags, actual_tags))
    total_tokens += len(actual_tags)

accuracy = total_correct / total_tokens

precision = {}
recall = {}

tags = set(all_actual) | set(all_predicted)

for tag in tags:
    tp = sum((p == tag and g == tag) for p, g in zip(all_predicted, all_actual))
    fp = sum((p == tag and g != tag) for p, g in zip(all_predicted, all_actual))
    fn = sum((p != tag and g == tag) for p, g in zip(all_predicted, all_actual))

    precision[tag] = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall[tag] = tp / (tp + fn) if (tp + fn) > 0 else 0

print("\nOverall Accuracy:", round(accuracy, 4))
print("Precision:", precision)
print("Recall:", recall)



Sentence: Will can spot Mary
Actual: ['NNP', 'MD', 'VB', 'NNP']
Predicted: ['NNP', 'MD', 'VB', 'NNP']

Sentence: Mary will see Spot
Actual: ['NNP', 'MD', 'VB', 'NNP']
Predicted: ['NNP', 'MD', 'VB', 'NNP']

Overall Accuracy: 1.0
Precision: {'VB': 1.0, 'NNP': 1.0, 'MD': 1.0}
Recall: {'VB': 1.0, 'NNP': 1.0, 'MD': 1.0}
